# Generate Synthetic 100M-Row Lance Dataset

Reads the original lance dataset, then creates a copy augmented to 100 million rows by sampling from the original data and multiplying each row by a random scale factor drawn from N(1.0, 0.1) clipped to [0.8, 1.2].

In [1]:
import shutil
import pathlib
from datetime import timedelta

import lance
import numpy as np
import pyarrow as pa

SOURCE_DIR = pathlib.Path("/Users/drew/code/hyrax/docs/pre_executed/results/20260406-134617-umap-2zH3")
DEST_DIR = pathlib.Path("/Users/drew/code/hyrax/docs/pre_executed/results/synthetic-1M_b")
LANCE_REL = pathlib.Path("lance_db/results.lance")  # relative path inside each result dir

TARGET_ROWS = 1_000_000
BATCH_SIZE = 500_000  # rows written per append call

rng = np.random.default_rng(42)

In [2]:
# ------------------------------------------------------------------
# 1. Inspect the source dataset
# ------------------------------------------------------------------
src_lance_path = SOURCE_DIR / LANCE_REL
src_ds = lance.dataset(str(src_lance_path))

print("Schema:")
print(src_ds.schema)
print(f"\nRow count: {src_ds.count_rows():,}")

orig_table = src_ds.to_table()
print("\nFirst 3 rows:")
print(orig_table.slice(0, 3).to_pydict())

Schema:
object_id: string
data: fixed_size_list<item: float>[2]
  child 0, item: float
-- schema metadata --
tensor_shape: '[2]'
tensor_dtype: 'float32'

Row count: 993

First 3 rows:
{'object_id': ['36407329666631333', '36407054788740841', '36412419202890477'], 'data': [[-1.940472960472107, 8.25619888305664], [-3.928934335708618, 3.4346723556518555], [-4.2150421142578125, 7.073315620422363]]}


In [3]:
# ------------------------------------------------------------------
# 2. Copy the entire source directory to the destination
# ------------------------------------------------------------------
if DEST_DIR.exists():
    print(f"Destination already exists: {DEST_DIR}")
    print("Remove it first if you want a clean run.")
else:
    shutil.copytree(SOURCE_DIR, DEST_DIR)
    print(f"Copied {SOURCE_DIR} -> {DEST_DIR}")

Copied /Users/drew/code/hyrax/docs/pre_executed/results/20260406-134617-umap-2zH3 -> /Users/drew/code/hyrax/docs/pre_executed/results/synthetic-1M_b


In [4]:
# ------------------------------------------------------------------
# 3. Identify numeric columns to scale; keep other columns as-is
# ------------------------------------------------------------------
schema = orig_table.schema
numeric_types = (
    pa.float16(),
    pa.float32(),
    pa.float64(),
    pa.int8(),
    pa.int16(),
    pa.int32(),
    pa.int64(),
    pa.uint8(),
    pa.uint16(),
    pa.uint32(),
    pa.uint64(),
)

# Also handle FixedSizeList (e.g. embedding vectors)
numeric_col_names = []
for field in schema:
    t = field.type
    if t in numeric_types:
        numeric_col_names.append(field.name)
    elif pa.types.is_floating(t) or pa.types.is_integer(t):
        numeric_col_names.append(field.name)
    elif pa.types.is_fixed_size_list(t):
        # check if the value type is numeric
        vt = t.value_type
        if pa.types.is_floating(vt) or pa.types.is_integer(vt):
            numeric_col_names.append(field.name)
    elif pa.types.is_large_list(t) or pa.types.is_list(t):
        vt = t.value_type
        if pa.types.is_floating(vt) or pa.types.is_integer(vt):
            numeric_col_names.append(field.name)

print("Numeric columns to be scaled:", numeric_col_names)
non_numeric = [f.name for f in schema if f.name not in numeric_col_names]
print("Non-numeric columns (copied as-is):", non_numeric)

Numeric columns to be scaled: ['data']
Non-numeric columns (copied as-is): ['object_id']


In [5]:
# ------------------------------------------------------------------
# 4. Helper: build one batch of synthetic rows
# ------------------------------------------------------------------
orig_np = {}  # column_name -> numpy array (for numeric cols)
for col in numeric_col_names:
    arr = orig_table.column(col)
    # FixedSizeList -> flatten to (n_orig, list_size) then back
    if (
        pa.types.is_fixed_size_list(arr.type)
        or pa.types.is_list(arr.type)
        or pa.types.is_large_list(arr.type)
    ):
        orig_np[col] = arr.to_pylist()  # list of lists
    else:
        orig_np[col] = arr.to_pylist()

orig_other = {}  # column_name -> python list (for non-numeric cols)
for col in non_numeric:
    orig_other[col] = orig_table.column(col).to_pylist()

n_orig = orig_table.num_rows
print(f"Original rows available for sampling: {n_orig:,}")


def make_batch(n_rows: int) -> pa.Table:
    """Sample n_rows from the original data with noise applied to numeric cols."""
    idx = rng.integers(0, n_orig, size=n_rows)

    # Scale factors: N(1.0, 0.1) clipped to [0.8, 1.2]
    scale = rng.normal(loc=1.0, scale=0.1, size=n_rows)
    scale = np.clip(scale, 0.8, 1.2).astype(np.float32)

    arrays = {}
    for col in numeric_col_names:
        field = schema.field(col)
        src = orig_np[col]
        sampled = [src[i] for i in idx]

        if (
            pa.types.is_fixed_size_list(field.type)
            or pa.types.is_list(field.type)
            or pa.types.is_large_list(field.type)
        ):
            # sampled is list of lists; apply per-row scale
            scaled = [[v * scale[r] for v in row] for r, row in enumerate(sampled)]
            arrays[col] = pa.array(scaled, type=field.type)
        else:
            vals = np.array(sampled, dtype=np.float64) * scale
            arrays[col] = pa.array(vals.tolist(), type=field.type)

    for col in non_numeric:
        src = orig_other[col]
        arrays[col] = pa.array([src[i] for i in idx], type=schema.field(col).type)

    # Preserve original column order
    ordered = [arrays[f.name] for f in schema]
    return pa.table({f.name: ordered[i] for i, f in enumerate(schema)}, schema=schema)

Original rows available for sampling: 993


In [6]:
# ------------------------------------------------------------------
# 5. Append synthetic rows to the destination lance dataset
# ------------------------------------------------------------------
dest_lance_path = DEST_DIR / LANCE_REL
dest_ds = lance.dataset(str(dest_lance_path))
existing_rows = dest_ds.count_rows()
print(f"Destination currently has {existing_rows:,} rows (original copy)")

rows_to_add = TARGET_ROWS - existing_rows
if rows_to_add <= 0:
    print(f"Already at or above {TARGET_ROWS:,} rows — nothing to do.")
else:
    print(f"Adding {rows_to_add:,} synthetic rows in batches of {BATCH_SIZE:,} ...")
    written = 0
    while written < rows_to_add:
        batch_n = min(BATCH_SIZE, rows_to_add - written)
        batch = make_batch(batch_n)
        lance.write_dataset(batch, str(dest_lance_path), mode="append")
        written += batch_n
        pct = 100 * written / rows_to_add
        print(f"  {written:>12,} / {rows_to_add:,}  ({pct:.1f}%)")

    final_count = lance.dataset(str(dest_lance_path)).count_rows()
    print(f"\nDone. Final row count: {final_count:,}")

    print(f"Total fragments: {len(lance.dataset(str(dest_lance_path)).get_fragments())}")
    # compact the dataset to reduce fragment count and improve read performance
    print("Compacting dataset to reduce fragment count...")
    lance.dataset(str(dest_lance_path)).optimize.compact_files()
    lance.dataset(str(dest_lance_path)).cleanup_old_versions(older_than=timedelta(0), delete_unverified=True)
    final_count = lance.dataset(str(dest_lance_path)).count_rows()
    print(f"After compaction fragment count: {len(lance.dataset(str(dest_lance_path)).get_fragments())}")

Destination currently has 993 rows (original copy)
Adding 999,007 synthetic rows in batches of 500,000 ...
       500,000 / 999,007  (50.0%)
       999,007 / 999,007  (100.0%)

Done. Final row count: 1,000,000
Total fragments: 3
Compacting dataset to reduce fragment count...
After compaction fragment count: 1


In [7]:
# ------------------------------------------------------------------
# 6. Quick sanity check
# ------------------------------------------------------------------
final_ds = lance.dataset(str(dest_lance_path))
print(f"Final dataset row count : {final_ds.count_rows():,}")
print(f"Schema: {final_ds.schema}")

# Show a sample from the tail to confirm synthetic data looks reasonable
sample = final_ds.to_table(offset=final_ds.count_rows() - 3)
print("\nLast 3 rows of synthetic data:")
print(sample.to_pydict())

Final dataset row count : 1,000,000
Schema: object_id: string
data: fixed_size_list<item: float>[2]
  child 0, item: float
-- schema metadata --
tensor_dtype: 'float32'
tensor_shape: '[2]'

Last 3 rows of synthetic data:
{'object_id': ['39912710174949548', '38549702303572846', '36424780118776069'], 'data': [[-1.60538911819458, 1.232132077217102], [-0.9437049031257629, 2.7323944568634033], [-2.532733917236328, 9.899369239807129]]}
